In [0]:
from pyspark.sql.types import StructField ,StructType,IntegerType,StringType,FloatType,DateType,TimestampType
import pyspark.sql.functions as F
catalog_name='ecommerce'

In [0]:
df_bronze=spark.table(f"{catalog_name}.bronze.brz_brands")
display(df_bronze.limit(5))

brand_code,brand_name,category_code,_souce_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z


In [0]:
#removing the leading and trailing spaces from brand name 
df_silver=df_bronze.withColumn('brand_name',F.trim(F.col('brand_name')))
display(df_silver.limit(5))

brand_code,brand_name,category_code,_souce_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z


In [0]:
#replacing all the symbol in the brandcode y making use of regular expression and replacing it with blank space 
df_silver=df_silver.withColumn('brand_code',F.regexp_replace(F.col('brand_code'),r'[^A-Za-z0-9]',' '))
display(df_silver.limit(10))

brand_code,brand_name,category_code,_souce_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
SKYL,SkyLink,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
VOLT,VoltEdge,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
PHTX,Photonix,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
URTL,UrbanTrail,APP,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
COTC,CottonClub,APP,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z


In [0]:
#to check the disticnt values from the category_code
display(df_silver.select('category_code').distinct())

category_code
CE
APP
HNK
BPC
BOOKS
BKS
GROCERY
GRCY
TOY
TOYS


In [0]:
anomalies = {
 "GROCERY": "GRCY",
 "BOOKS": "BKS",
 "TOYS": "TOY"
}
df_silver=df_silver.replace(anomalies,subset='category_code') #subset=category_code means do the replacement in the category_code column only 
display(df_silver.limit(10))

brand_code,brand_name,category_code,_souce_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
SKYL,SkyLink,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
VOLT,VoltEdge,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
PHTX,Photonix,CE,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
URTL,UrbanTrail,APP,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z
COTC,CottonClub,APP,dbfs:/Volumes/ecommerce/source_data/raw/brands/brands.csv,2026-03-11T12:07:01.112Z


In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_brands)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")

df_bronze.show(10)

+-------------+--------------------+--------------------+--------------------+
|category_code|       category_name|        _ingested_at|        _source_file|
+-------------+--------------------+--------------------+--------------------+
|           ce|         Electronics|2026-03-11 12:11:...|dbfs:/Volumes/eco...|
|          app|             Apparel|2026-03-11 12:11:...|dbfs:/Volumes/eco...|
|          hnk|      Home & Kitchen|2026-03-11 12:11:...|dbfs:/Volumes/eco...|
|          bpc|Beauty & Personal...|2026-03-11 12:11:...|dbfs:/Volumes/eco...|
|          bks|               Books|2026-03-11 12:11:...|dbfs:/Volumes/eco...|
|         grcy|             Grocery|2026-03-11 12:11:...|dbfs:/Volumes/eco...|
|          toy|        Toys & Games|2026-03-11 12:11:...|dbfs:/Volumes/eco...|
|          spt|   Sports & Outdoors|2026-03-11 12:11:...|dbfs:/Volumes/eco...|
|          app|             Apparel|2026-03-11 12:11:...|dbfs:/Volumes/eco...|
|         grcy|             Grocery|2026-03-11 12:11

In [0]:
df_duplicates=df_bronze.groupby("category_code").count().filter(F.col("count")>1)
display(df_duplicates)

category_code,count
app,2
grcy,2


In [0]:
df_silver=df_bronze.dropDuplicates(['category_code'])
display(df_silver)

category_code,category_name,_ingested_at,_source_file
ce,Electronics,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
app,Apparel,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
hnk,Home & Kitchen,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
bpc,Beauty & Personal Care,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
bks,Books,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
grcy,Grocery,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
toy,Toys & Games,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
spt,Sports & Outdoors,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv


In [0]:
#converting the category_code in the upper case 
df_silver=df_silver.withColumn('category_code',F.upper(F.col("category_code")))
display(df_silver)
#withColumn is the function the pyspark that cretes or modify the existing table 


category_code,category_name,_ingested_at,_source_file
CE,Electronics,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
APP,Apparel,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
HNK,Home & Kitchen,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
BPC,Beauty & Personal Care,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
BKS,Books,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
GRCY,Grocery,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
TOY,Toys & Games,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv
SPT,Sports & Outdoors,2026-03-11T12:11:27.788Z,dbfs:/Volumes/ecommerce/source_data/raw/category/category.csv


In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_category)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_category")

Products

In [0]:
df_bronze=spark.read.table(f"{catalog_name}.bronze.brz_products")

#get the count of the rows and columns
row_count,column_count=df_bronze.count(),len(df_bronze.columns)

#print the results
print(f"Row Count:{row_count}")
print(f"Column count:{column_count}")

Row Count:50000
Column count:14


In [0]:
# Check weight_grams column
df_bronze.select("weight_grams").show(5, truncate=False)

+------------+
|weight_grams|
+------------+
|305g        |
|682g        |
|243g        |
|225g        |
|455g        |
+------------+
only showing top 5 rows


In [0]:
# replace 'g' with ''
df_silver = df_bronze.withColumn(
    "weight_grams",
    F.regexp_replace(F.col("weight_grams"), "g", "").cast(IntegerType())
)
df_silver.select("weight_grams").show(5, truncate=False)

+------------+
|weight_grams|
+------------+
|305         |
|682         |
|243         |
|225         |
|455         |
+------------+
only showing top 5 rows


In [0]:
df_silver.select("length_cm").show(3)

+---------+
|length_cm|
+---------+
|     22,2|
|     18,2|
|     18,2|
+---------+
only showing top 3 rows


In [0]:
# replace , with .
df_silver = df_silver.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"), ",", ".").cast(FloatType())
)
df_silver.select("length_cm").show(3)

+---------+
|length_cm|
+---------+
|     22.2|
|     18.2|
|     18.2|
+---------+
only showing top 3 rows


In [0]:
df_silver.select("category_code", "brand_code").show(2)

+-------------+----------+
|category_code|brand_code|
+-------------+----------+
|          hnk|      stcr|
|          hnk|      hmns|
+-------------+----------+
only showing top 2 rows


In [0]:
# convert category_code and brand_code to upper case
df_silver = df_silver.withColumn(
    "category_code",
    F.upper(F.col("category_code"))
).withColumn(
    "brand_code",
    F.upper(F.col("brand_code"))
)
df_silver.select("category_code", "brand_code").show(2)

+-------------+----------+
|category_code|brand_code|
+-------------+----------+
|          HNK|      STCR|
|          HNK|      HMNS|
+-------------+----------+
only showing top 2 rows


In [0]:
df_silver.select("material").distinct().show()

+---------+
| material|
+---------+
|    Coton|
|    Steel|
|     Wood|
|    Ruber|
|  Plastic|
|Polyester|
|    Glass|
|  Alumium|
|    Paper|
|  Leather|
+---------+



In [0]:
# Fix spelling mistakes which have been encountered while we checked the distinct values 
df_silver = df_silver.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
     .when(F.col("material") == "Alumium", "Aluminum")
     .when(F.col("material") == "Ruber", "Rubber")
     .otherwise(F.col("material"))
)
df_silver.select("material").distinct().show()    

+---------+
| material|
+---------+
|   Cotton|
|    Steel|
|     Wood|
|   Rubber|
|  Plastic|
|Polyester|
|    Glass|
| Aluminum|
|    Paper|
|  Leather|
+---------+



In [0]:
#checking for the negative values in the rating column because rating cannot contains negative values 
df_silver.filter(F.col('rating_count')<0).select("rating_count").show(3)


+------------+
|rating_count|
+------------+
|          -4|
|          -2|
|          -2|
+------------+
only showing top 3 rows


In [0]:
# Convert negative rating_count to positive
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")))
     .otherwise(F.lit(0))  # if null, replace with 0
)

In [0]:
# Check final cleaned data

df_silver.select(
    "weight_grams",
    "length_cm",
    "category_code",
    "brand_code",
    "material",
    "rating_count"
).show(10, truncate=False)

+------------+---------+-------------+----------+---------+------------+
|weight_grams|length_cm|category_code|brand_code|material |rating_count|
+------------+---------+-------------+----------+---------+------------+
|305         |22.2     |HNK          |STCR      |Cotton   |0           |
|682         |18.2     |HNK          |HMNS      |Steel    |1           |
|243         |18.2     |CE           |NOVW      |Wood     |0           |
|225         |17.6     |APP          |URTL      |Rubber   |50          |
|455         |27.2     |GRCY         |GGRN      |Rubber   |4           |
|232         |28.0     |BPC          |SLKE      |Plastic  |0           |
|507         |27.2     |CE           |VOLT      |Plastic  |5           |
|261         |27.7     |APP          |CBLT      |Polyester|0           |
|59          |12.5     |SPT          |ARFT      |Plastic  |11          |
|238         |10.7     |APP          |MOSA      |Polyester|6           |
+------------+---------+-------------+----------+--

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_dim_products)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_products")

Customer

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(10)

Row count: 300000
Column count: 7
+----------------+--------------+------------+--------------+-----+--------------------+--------------------+
|     customer_id|         phone|country_code|       country|state|           file_name|    ingest_timestamp|
+----------------+--------------+------------+--------------+-----+--------------------+--------------------+
|CUST000000000001|917280033536.0|          IN|         India|   MH|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000002|619489725433.0|          AU|     Australia|  VIC|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000003|919390066524.0|          IN|         India|   TN|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000004|917073741793.0|          IN|         India|   TN|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000005|618478772532.0|          AU|     Australia|   WA|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000006|916441718520.0|          IN|         India|   GJ|dbfs:/Volumes/eco..

In [0]:
#handling the null values in the cutsomer_id column 
null_count = df_bronze.filter(F.col("customer_id").isNull()).count()
null_count

300

In [0]:
# There are 300 null values in customer_id column. Display some of those
df_bronze.filter(F.col("customer_id").isNull()).show(3)

+-----------+--------------+------------+-------+-----+--------------------+--------------------+
|customer_id|         phone|country_code|country|state|           file_name|    ingest_timestamp|
+-----------+--------------+------------+-------+-----+--------------------+--------------------+
|       NULL|918187043562.0|          IN|  India|   DL|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|       NULL|917517243052.0|          IN|  India|   DL|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|       NULL|          NULL|          IN|  India|   GJ|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
+-----------+--------------+------------+-------+-----+--------------------+--------------------+
only showing top 3 rows


In [0]:
# Drop rows where 'customer_id' is null
#we will be dropping the rows with the null values and checking the count of the rows after dropping the rows 
df_silver = df_bronze.dropna(subset=["customer_id"]) #subset="customer_id" means it will happen only in the cutsomer_id column 

# Get row count
row_count = df_silver.count()
print(f"Row count after droping null values: {row_count}")

Row count after droping null values: 299700


In [0]:
#handling the missing values in the phone column 
null_count=df_silver.filter(F.col("phone").isNull()).count()
print(f"{null_count}")

29964


In [0]:
df_silver.filter(F.col("phone").isNull()).show(10)

+----------------+-----+------------+-------------+-----+--------------------+--------------------+
|     customer_id|phone|country_code|      country|state|           file_name|    ingest_timestamp|
+----------------+-----+------------+-------------+-----+--------------------+--------------------+
|CUST000000000007| NULL|          IN|        India|   MH|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000010| NULL|          IN|        India|   RJ|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000026| NULL|          IN|        India|   WB|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000032| NULL|          US|United States|   NJ|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000070| NULL|          IN|        India|   TS|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000081| NULL|          AU|    Australia|  VIC|dbfs:/Volumes/eco...|2026-03-11 12:13:...|
|CUST000000000103| NULL|          US|United States|   CA|dbfs:/Volumes/eco...|2026-03-11 12:13:...|


In [0]:
#we will be filling the null values with not available 
df_silver=df_silver.fillna("Not Available",subset=["phone"])

#lets o a sanity check to have a look at athe data after filtering it wheter i contains any null values 
df_silver.filter(F.col("phone").isNull()).show(10)

+-----------+-----+------------+-------+-----+---------+----------------+
|customer_id|phone|country_code|country|state|file_name|ingest_timestamp|
+-----------+-----+------------+-------+-----+---------+----------------+
+-----------+-----+------------+-------+-----+---------+----------------+



In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_customers)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

Calender/date

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(3)

Row count: 95
Column count: 7
+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+--------+-------+------------+--------------------+--------------------+
|01-08-2025|2025|  friday|      3|         -31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|02-08-2025|2025|SATURDAY|      3|         -31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|03-08-2025|2025|  SUNDAY|      3|         -31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 3 rows


In [0]:
df_bronze.printSchema()

root
 |-- date: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [0]:
from pyspark.sql.functions import to_date


# Convert the string column to a date type
df_silver = df_bronze.withColumn("date", to_date(df_bronze["date"], "dd-MM-yyyy"))

In [0]:
print(df_silver.printSchema())

df_silver.show(5)

root
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- week_of_year: integer (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

None
+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  friday|      3|         -31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-02|2025|SATURDAY|      3|         -31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-03|2025|  SUNDAY|      3|         -31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-04|2025|  MONDAY|      3|         -32|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-05|2025| TUESDAY|      3|         -32|2026-03-11 12:15:...|dbfs

In [0]:
#find the duplocates record in the dataframe
duplicates=df_silver.groupby('date').count().filter("count>1")
print("dupliactes records are:",duplicates.count())

dupliactes records are: 3


In [0]:
# Remove duplicate rows
df_silver = df_silver.dropDuplicates(['date'])

# Get row count
row_count = df_silver.count()

print("Rows After removing Duplicates: ", row_count)

Rows After removing Duplicates:  92


In [0]:
# Capitalize first letter of each word in day_name
df_silver = df_silver.withColumn("day_name", F.initcap(F.col("day_name")))

df_silver.show(5)

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  Friday|      3|         -31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-02|2025|Saturday|      3|         -31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-03|2025|  Sunday|      3|         -31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-04|2025|  Monday|      3|         -32|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-05|2025| Tuesday|      3|         -32|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 5 rows


In [0]:
df_silver=df_silver.withColumn("week_of_year",F.abs(F.col("week_of_year")))
df_silver.show(10)

+----------+----+---------+-------+------------+--------------------+--------------------+
|      date|year| day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+---------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|   Friday|      3|          31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-02|2025| Saturday|      3|          31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-03|2025|   Sunday|      3|          31|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-04|2025|   Monday|      3|          32|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-05|2025|  Tuesday|      3|          32|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-06|2025|Wednesday|      3|          32|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-07|2025| Thursday|      3|          32|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-08|2025|   Friday|      3|          32|2026-03-11 12:15:...|dbfs:/Volumes/eco...|

In [0]:
df_silver = df_silver.withColumn("quarter", F.concat_ws("", F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))))

df_silver = df_silver.withColumn("week_of_year", F.concat_ws("-", F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year"))))

df_silver.show(3)

+----------+----+--------+-------+------------+--------------------+--------------------+
|      date|year|day_name|quarter|week_of_year|        _ingested_at|        _source_file|
+----------+----+--------+-------+------------+--------------------+--------------------+
|2025-08-01|2025|  Friday|Q3-2025| Week31-2025|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-02|2025|Saturday|Q3-2025| Week31-2025|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
|2025-08-03|2025|  Sunday|Q3-2025| Week31-2025|2026-03-11 12:15:...|dbfs:/Volumes/eco...|
+----------+----+--------+-------+------------+--------------------+--------------------+
only showing top 3 rows


In [0]:
# Rename a column
df_silver = df_silver.withColumnRenamed("week_of_year", "week")

In [0]:
# Write raw data to the silver layer (catalog: ecommerce, schema: silver, table: slv_calendar)
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")